# 01 · Data validation

Checks the raw delivery data before any analysis. **Nothing is dropped, imputed or capped** — every check passes.

Supports deck slide **A5 (Data sources & validation)**.

In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from common import load, customers, weekly_new, CHANNEL_ORDER, LAST_WEEK_IDX
pd.set_option("display.width", 140)
df = load()

In [2]:
df.head()

,delivery_id,customer_id,delivery_week,country,box_size,acquisition_channel,acquisition_week,gross_revenue,net_revenue,discount_amount,discount_type,is_first_delivery,boxes_in_delivery,customer_lifetime_deliveries,d_idx,a_idx,d_year,a_year,iso_wk,tenure
0,D000001,C10001,2023-W01,US,2,Paid Social,2023-W01,59.99,36.37,23.62,First Order,1,1,1,0,0,2023,2023,1,0
1,D000002,C10001,2023-W02,US,2,Paid Social,2023-W01,59.99,55.30,4.69,Loyalty,0,1,2,1,0,2023,2023,2,1
2,D000003,C10002,2023-W01,US,6,Paid Social,2023-W01,119.99,78.98,41.01,First Order,1,1,1,0,0,2023,2023,1,0
3,D000004,C10002,2023-W02,US,6,Paid Social,2023-W01,119.99,119.99,0.00,NaN,0,1,2,1,0,2023,2023,2,1
4,D000005,C10002,2023-W03,US,6,Paid Social,2023-W01,119.99,119.99,0.00,NaN,0,1,3,2,0,2023,2023,3,2


In [3]:
checks = {
    "rows": len(df),
    "customers": df.customer_id.nunique(),
    "weeks": df.delivery_week.nunique(),
    "countries": df.country.unique().tolist(),
    "duplicate delivery_id": int(df.delivery_id.duplicated().sum()),
    "duplicate customer-week": int(df.duplicated(["customer_id", "delivery_week"]).sum()),
    "max |gross - discount - net|": float((df.gross_revenue - df.discount_amount - df.net_revenue).abs().max()),
    "every customer has exactly 1 first delivery": bool((df.groupby("customer_id").is_first_delivery.sum() == 1).all()),
    "acquisition_week constant per customer": bool((df.groupby("customer_id").acquisition_week.nunique() == 1).all()),
    "first delivery week == acquisition week": bool((df[df.is_first_delivery == 1].delivery_week
                                                     == df[df.is_first_delivery == 1].acquisition_week).all()),
    "discount_type missing exactly when discount == 0": bool((df.discount_type.isna() == (df.discount_amount == 0)).all()),
    "negative revenue / discount values": int((df[["gross_revenue", "net_revenue", "discount_amount"]] < 0).sum().sum()),
    "ISO week 53 present": int(df.delivery_week.str.endswith("W53").sum()),
}
for k, v in checks.items():
    print(f"{k:50s} {v}")

rows                                               53824
customers                                          9062
weeks                                              156
countries                                          ['US']
duplicate delivery_id                              0
duplicate customer-week                            0
max |gross - discount - net|                       2.842170943040401e-14
every customer has exactly 1 first delivery        True
acquisition_week constant per customer             True
first delivery week == acquisition week            True
discount_type missing exactly when discount == 0   True
negative revenue / discount values                 0
ISO week 53 present                                0


### Notes on the fields

- **`discount_type`** holds the literal text `None` for undiscounted deliveries; pandas reads it as missing. It is missing on exactly the 34,518 zero-discount rows, so it means "no discount", not lost data.
- **`customer_lifetime_deliveries`** is a running counter (1, 2, 3 …) per delivery, not a lifetime total. Lifetime tiers therefore use each customer's count of delivery rows.
- **Gross revenue** takes only six values (three box prices × one or two boxes), so there are no outliers to treat. Two-box deliveries are genuine grouped orders and stay in.

In [4]:
print("Gross revenue values by box size and boxes in delivery:")
df.groupby(["box_size", "boxes_in_delivery"]).gross_revenue.agg(["min", "max", "count"])

Gross revenue values by box size and boxes in delivery:


min     max  count
box_size boxes_in_delivery                       
2        1                   59.99   59.99  16035
         2                  119.98  119.98    547
4        1                   89.99   89.99  21328
         2                  179.98  179.98    775
6        1                  119.99  119.99  14572
         2                  239.98  239.98    567

In [5]:
s = df.sort_values(["customer_id", "d_idx"])
running = s.groupby("customer_id").cumcount() + 1
print("customer_lifetime_deliveries equals the running delivery count on every row:",
      bool((running == s.customer_lifetime_deliveries).all()))

customer_lifetime_deliveries equals the running delivery count on every row: True
